# Module 5: Q2, off-plan vs ready (Python)

**Business question:** is off-plan still taking share from ready homes, and **do buyers pay a premium per sq ft for off-plan apartments in the same community and unit type?** If so, how big is it, how sure can we be, and is it changing?

**Input:** `data/extracts/q2_apartments.parquet` (from Module 5a): 582,612 apartment sales, 2021 to Oct 2026, outliers excluded
**Output:** results CSVs in `data/gold/`, two charts in `images/`, and your findings

**Time:** 2–3 hours. **New skills:** pandas `groupby`/`unstack`, confounding and like-for-like comparison, weighted medians, the Mann–Whitney U test, p-values and effect sizes, multiple-testing correction, bootstrap confidence intervals, a fixed-effects regression, and matplotlib charts for a README.

## 🧠 Why we're doing this: three questions statistics answers

Module 4 was *descriptive*: what happened. Q2 needs *inferential* statistics, because a difference in two medians raises three questions any sharp stakeholder will ask:

1. **Is it real, or could it be chance?** → a **hypothesis test** and its **p-value**
2. **How big is it?** → an **effect size** (a p-value never tells you this)
3. **How sure are we about the size?** → a **confidence interval**

With 580k sales, almost *any* difference will be "statistically significant". So questions 2 and 3 matter more. You'll hear this in interviews: *"significant isn't the same as important."*

## Setup
Run this first. If an import fails, your environment isn't set up yet: see `modules/05_vscode_setup.md`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

# Paths work whether VS Code runs the notebook from the repo root or from python/
REPO = Path.cwd().parent if Path.cwd().name == "python" else Path.cwd()
DATA_FILE = REPO / "data" / "extracts" / "q2_apartments.parquet"
OUT_DIR   = REPO / "data" / "gold"
IMG_DIR   = REPO / "images"
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(exist_ok=True)

MIN_PER_GROUP = 30      # minimum off-plan AND ready sales in a group before comparing them
N_BOOT        = 2_000   # bootstrap resamples
SEED          = 42      # makes random results reproducible

pd.set_option("display.float_format", "{:,.1f}".format)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

def check(label, actual, expected):
    print(f"{'✅' if actual == expected else '❌'} {label}: got {actual}, expected {expected}")

def check_close(label, actual, expected, tolerance=0.02):
    ok = actual is not None and abs(actual - expected) <= tolerance * abs(expected)
    f = ",.0f" if isinstance(expected, int) else ",.1f"
    print(f"{'✅' if ok else '❌'} {label}: got {actual:{f}}, expected about {expected:{f}} (±{tolerance:.0%})")

def year_labels(years):
    """Label the last year as year-to-date, so a partial year is never read as a full one."""
    last = max(years)
    return [f"{y}\n(Jan–Oct)" if y == last else str(y) for y in years]

print("Repo folder:", REPO)
print("Data file found:", DATA_FILE.exists())

## Step 1: Load the extract

🧠 **Why we're doing this**
`pd.read_parquet` reads the file *with its types*: no `dtype=` arguments, no date parsing. Compare that with all the casting in Module 2. `df.info()` is pandas' `printSchema()`.

✏️ **TODO 1:** read `DATA_FILE` into `df`.

In [ ]:
# TODO 1: load the Parquet file
df = ___

df.info()
check_close("Rows", len(df), 582612)
df.head()

## Step 2: Is off-plan still taking share? (Q2, part 1)

🧠 **Why we're doing this**
The first half of Q2 is descriptive. A neat pandas trick: comparing a column to a value gives `True`/`False`, and the **mean of booleans is the proportion that are `True`**. It's the same idea as `AVG(CAST(is_off_plan AS INT))` in SQL.

`.groupby("year")[col].mean()` is `GROUP BY year` with `AVG(col)`.

✏️ **TODO 2:** group `df` by `year` and take the mean of `is_off_plan`, times 100.

In [ ]:
df["is_off_plan"] = df["market"] == "Off-plan"

# TODO 2: off-plan share of apartment sales by year (%)
share = ___

print(share.round(1))
check_close("Off-plan share 2021 (%)", share[2021], 47.5, tolerance=0.01)
check_close("Off-plan share 2025 (%)", share[2025], 74.5, tolerance=0.01)

ax = share.plot(kind="bar", color="#4C72B0", rot=0)
ax.set_xticklabels(year_labels(share.index))
ax.set(title="Off-plan share of apartment sales", xlabel="", ylabel="% of sales", ylim=(0, 100))
ax.bar_label(ax.containers[0], fmt="%.0f%%")
plt.tight_layout(); plt.show()

## Step 3: The naive premium

🧠 **Why we're doing this**
The obvious first answer: compare the median price per sq ft of *all* off-plan apartments with *all* ready apartments, year by year.

`.unstack("market")` turns the `market` values into **columns**. It's a pivot: rows = years, columns = Off-plan / Ready. (Its opposite, `.stack()`, is unpivot.)

✏️ **TODO 3:** the naive premium in %: `(Off-plan / Ready − 1) × 100`.

In [ ]:
naive = df.groupby(["year", "market"])["price_per_sqft"].median().unstack("market")

# TODO 3: naive premium (%)
naive["naive_premium_pct"] = ___

print(naive)
check_close("Naive premium 2025 (%)", naive.loc[2025, "naive_premium_pct"], 33.2, tolerance=0.02)
check_close("Naive premium 2026 (%)", naive.loc[2026, "naive_premium_pct"], 23.9, tolerance=0.02)

On the naive numbers, the off-plan premium **collapsed**, from about 56% in 2021 to 24% in 2026. Is that the story? Not yet. Keep reading.

## Step 4: Why the naive comparison can mislead (confounding)

🧠 **Why we're doing this**
Off-plan and ready apartments aren't sold in the same *places*. If off-plan sales are concentrated in cheaper communities, the off-plan median is dragged down even if every off-plan flat costs more than a ready flat next door. A third variable (here, **location**) that affects both "which market" and "price" is called a **confounder**.

Run this: the busiest communities for each market in 2025, with the *ready* median in that community as a yardstick of how expensive the area is.

In [ ]:
d25 = df[df["year"] == 2025]
ready_level = d25[d25["market"] == "Ready"].groupby("community")["price_per_sqft"].median()

busiest = (d25.groupby(["market", "community"]).size().rename("sales").reset_index()
              .sort_values("sales", ascending=False).groupby("market").head(6))
busiest["ready_median_in_community"] = busiest["community"].map(ready_level).round(0)
busiest

Off-plan volume sits heavily in places like **Dubai Land Residence Complex** and **Motor City**, where ready homes sell for well under AED 1,000 per sq ft. The ready list includes **Downtown** and **Dubai Marina**, two of the most expensive areas. So the naive comparison partly measures *where* each market sells, not *what* buyers pay for off-plan.

**The fix is a like-for-like comparison:** compare off-plan with ready **inside the same community, bedroom band and year**, then combine those comparisons. It's the same logic as Module 4's mix check, applied systematically.

🎤 **Interview angle:** *"How do you know the difference isn't caused by something else?"* → *"I controlled for the obvious confounders, location and unit type, by comparing within matched groups, and cross-checked with a fixed-effects regression."*

## Step 5: Like-for-like groups

🧠 **Why we're doing this**
We build one row per **year × community × bedroom band**, with the off-plan median, ready median and the count of each. A group only counts if it has at least `MIN_PER_GROUP` (30) sales on **both** sides. That's the same minimum-sample logic as Module 4.

Two pandas details:
- `.agg(median="median", n="size")` names the outputs, like `AS` in SQL.
- After `unstack`, column names are pairs like `("median", "Off-plan")`. The list comprehension flattens them to `median_off_plan`, `n_ready` and so on.

✏️ **TODO 5a:** keep groups where `n_off_plan` **and** `n_ready` are both `>= MIN_PER_GROUP` (use `&`, with each condition in brackets).
✏️ **TODO 5b:** the premium in %, per group.

In [ ]:
cells = (df.groupby(["year", "community", "bedroom_band", "market"])["price_per_sqft"]
           .agg(median="median", n="size")
           .unstack("market"))
cells.columns = [f"{stat}_{m.lower().replace('-', '_')}" for stat, m in cells.columns]
cells = cells.reset_index()

# TODO 5a: groups with enough sales on both sides
comparable = cells[___].copy()

# TODO 5b: premium per group (%)
comparable["premium_pct"] = ___
comparable["weight"] = comparable["n_off_plan"] + comparable["n_ready"]

c25 = comparable[comparable["year"] == 2025]
check("Comparable groups in 2025", len(c25), 120)
check_close("Median premium across 2025 groups (%)", c25["premium_pct"].median(), 29.1)
check_close("% of 2025 groups where off-plan is dearer", (c25["premium_pct"] > 0).mean() * 100, 93.3, tolerance=0.01)
c25.sort_values("weight", ascending=False).head(10)

🤔 **Think about it:** a group with 0 ready sales has `NaN` in `n_ready` after `unstack`. Does your filter keep or drop it, and why?

## Step 6: One market-wide number per year (weighted median)

🧠 **Why we're doing this**
We now have ~120 premiums for 2025, one per group. To summarise them, a plain median treats a group of 60 sales the same as JVC 1-bed with 9,000. A **weighted median** gives each group a say in proportion to its sales. It's the premium paid by the *typical buyer*, not the typical group.

How a weighted median works: sort the values, add up the weights, and pick the value where the running total crosses half. The helper below does exactly that. Read its four lines.

✏️ **TODO 6:** inside the loop, call `weighted_median` on the group's `premium_pct` and `weight`.

In [ ]:
def weighted_median(values, weights):
    """The value where the cumulative weight first reaches half of the total."""
    order = np.argsort(values)
    v, w = np.asarray(values)[order], np.asarray(weights)[order]
    cumulative = np.cumsum(w)
    return v[np.searchsorted(cumulative, cumulative[-1] / 2)]

rows = []
for year, g in comparable.groupby("year"):
    rows.append({
        "year": year,
        "groups": len(g),
        "sales_covered": int(g["weight"].sum()),
        # TODO 6
        "like_for_like_premium_pct": ___,
    })
lfl = pd.DataFrame(rows).set_index("year")
lfl["naive_premium_pct"] = naive["naive_premium_pct"]
lfl["coverage_pct"] = lfl["sales_covered"] / df.groupby("year").size() * 100

print(lfl)
check_close("Like-for-like premium 2021 (%)", lfl.loc[2021, "like_for_like_premium_pct"], 19.1)
check_close("Like-for-like premium 2025 (%)", lfl.loc[2025, "like_for_like_premium_pct"], 30.4)
check_close("Like-for-like premium 2026 (%)", lfl.loc[2026, "like_for_like_premium_pct"], 32.4)

In [ ]:
ax = lfl[["naive_premium_pct", "like_for_like_premium_pct"]].plot(marker="o", color=["#B0B0B0", "#C44E52"])
ax.set(title="Off-plan premium per sq ft over ready apartments", xlabel="", ylabel="Premium (%)", ylim=(0, None))
ax.legend(["Naive (all off-plan vs all ready)", "Like-for-like (same community and bedrooms)"], frameon=False)
ax.set_xticks(lfl.index, year_labels(lfl.index))
plt.tight_layout()
plt.savefig(IMG_DIR / "q2_naive_vs_like_for_like.png", dpi=150)
plt.show()

**This is the main finding of Q2.** The two lines move in **opposite directions**:
- **Naive:** the gap shrinks, from about 56% to 24%.
- **Like for like:** the premium *grows*, from about 19% to 32%.

The naive line falls because off-plan expanded into cheaper communities (Step 4). *Within* each community, buyers are paying **more** for off-plan relative to ready than they did in 2021. Reporting the naive number would have produced exactly the wrong headline.

When two levels of aggregation tell opposite stories, statisticians call it **Simpson's paradox**. Now you have a real example of it from your own data.

## Step 7: One hypothesis test, JVC 1-bed apartments in 2025

🧠 **Why we're doing this**
Before testing 120 groups, understand one. JVC 1-bed is the busiest group (6,218 off-plan and 2,886 ready sales).

**The hypotheses:**
- **H₀ (null):** off-plan and ready prices come from the same distribution. Any gap is chance.
- **H₁ (alternative):** they don't.

**The p-value** is the probability of seeing a gap at least this big *if H₀ were true*. A small p-value (conventionally < 0.05) means "chance is an unlikely explanation". It is **not** the probability that H₀ is true, and it says **nothing about how big** the gap is.

**Why Mann–Whitney U, not a t-test?** The t-test compares *means* and assumes roughly normal data. Prices per sq ft are right-skewed (across 2025 apartments the mean is ~AED 1,900 and the median ~1,730), and we've been working with *medians* throughout. Mann–Whitney compares **ranks**, so it needs no normality assumption and isn't thrown off by extreme values.

**Effect size: the common-language effect size (CLES).** `U / (n₁ × n₂)` is the probability that a randomly chosen off-plan sale is priced higher per sq ft than a randomly chosen ready sale. 50% means no difference. It's an effect size you can say to a non-technical manager.

✏️ **TODO 7:** run `stats.mannwhitneyu(off, ready, alternative="two-sided")`.

In [ ]:
jvc = df[(df["year"] == 2025) & (df["community"] == "Jumeirah Village Circle") & (df["bedroom_band"] == "1 BR")]
off   = jvc.loc[jvc["market"] == "Off-plan", "price_per_sqft"].to_numpy()
ready = jvc.loc[jvc["market"] == "Ready",    "price_per_sqft"].to_numpy()

# TODO 7: Mann-Whitney U test
result = ___

cles = result.statistic / (len(off) * len(ready))
premium = (np.median(off) / np.median(ready) - 1) * 100

print(f"Off-plan median AED {np.median(off):,.0f} vs ready AED {np.median(ready):,.0f} per sq ft ({premium:+.1f}%)")
print(f"Mann-Whitney U = {result.statistic:,.0f}, p-value = {result.pvalue:.1e}")
print(f"CLES: a random off-plan sale is dearer per sq ft than a random ready sale {cles:.1%} of the time")
check_close("CLES (%)", cles * 100, 71.5, tolerance=0.01)
check("p-value below 0.001", bool(result.pvalue < 0.001), True)

fig, ax = plt.subplots()
bins = np.linspace(500, 2500, 60)
ax.hist(ready, bins=bins, alpha=0.6, label=f"Ready (n={len(ready):,})", color="#4C72B0")
ax.hist(off,   bins=bins, alpha=0.6, label=f"Off-plan (n={len(off):,})", color="#C44E52")
ax.set(title="JVC 1-bed apartments, 2025: price per sq ft", xlabel="AED per sq ft", ylabel="Sales")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

The p-value is astronomically small, which is unsurprising with ~9,000 sales. The **useful** numbers are the premium (~17%) and the CLES (~72%): a clear but not overwhelming gap, with plenty of overlap in the histogram. That overlap is why the CLES isn't 100%.

## Step 8: Testing all 2025 groups, and the multiple-testing trap

🧠 **Why we're doing this**
Run 120 tests at the 5% level and, **even if no real differences existed**, you'd expect about 6 "significant" results by pure chance. The more tests, the more false alarms. This is the **multiple comparisons problem**.

The **Benjamini–Hochberg (BH)** correction controls the **false discovery rate**: among the results you call significant, at most about 5% are expected to be false alarms. It's the standard choice when you run many tests and want to keep most real findings. (The stricter Bonferroni divides 0.05 by the number of tests.)

The loop runs one Mann–Whitney test per group. A dictionary of groups (`groups[(community, band)]`) avoids re-filtering the whole DataFrame 120 times.

✏️ **TODO 8:** `multipletests(tests["p_value"], alpha=0.05, method="fdr_bh")`. It returns four things. We need the first two: which tests are rejected, and the adjusted p-values.

In [ ]:
groups = {key: g for key, g in d25.groupby(["community", "bedroom_band"])}
tests = comparable[comparable["year"] == 2025].copy()

p_values = []
for _, row in tests.iterrows():
    g = groups[(row["community"], row["bedroom_band"])]
    p = stats.mannwhitneyu(g.loc[g["market"] == "Off-plan", "price_per_sqft"],
                           g.loc[g["market"] == "Ready",    "price_per_sqft"],
                           alternative="two-sided").pvalue
    p_values.append(p)
tests["p_value"] = p_values

# TODO 8: Benjamini-Hochberg correction
reject, p_adjusted, _, _ = ___
tests["p_adjusted"] = p_adjusted
tests["significant"] = reject

print(f"Groups tested:                          {len(tests)}")
print(f"Significant before correction (p<0.05): {(tests['p_value'] < 0.05).sum()}")
print(f"Significant after BH correction:        {tests['significant'].sum()}")
print(f"  of which off-plan CHEAPER than ready: {(tests['significant'] & (tests['premium_pct'] < 0)).sum()}")
check("Significant after BH", int(tests["significant"].sum()), 116)

In [ ]:
# The significant groups where off-plan is CHEAPER: the exceptions worth a sentence in your findings
tests[tests["significant"] & (tests["premium_pct"] < 0)][
    ["community", "bedroom_band", "median_off_plan", "median_ready", "premium_pct", "n_off_plan", "n_ready"]]

## Step 9: How sure are we? Bootstrap confidence intervals

🧠 **Why we're doing this**
A **95% confidence interval** is a range built so that, if we repeated the whole study many times, 95% of such ranges would contain the true value. Narrow means precise; wide means uncertain.

For medians and weighted medians there's no simple formula, so we use the **bootstrap**:
1. Pretend your sample *is* the population.
2. Draw a new sample of the same size **with replacement** (some rows twice, some not at all).
3. Recalculate the statistic. Repeat 2,000 times.
4. The middle 95% of those 2,000 results (the 2.5th to 97.5th percentiles) is your interval.

It's brute force instead of algebra, which is exactly what computers are good at. `rng = np.random.default_rng(SEED)` makes the randomness **reproducible**: same seed, same answer, every run.

**9a (one group):** resample JVC 1-bed's off-plan and ready sales separately.
✏️ **TODO 9a:** draw a resample of `off` with `rng.choice(off, size=len(off), replace=True)`. The `ready` line shows the pattern.

In [ ]:
rng = np.random.default_rng(SEED)
boot = np.empty(N_BOOT)
for b in range(N_BOOT):
    # TODO 9a: resample the off-plan prices
    off_b   = ___
    ready_b = rng.choice(ready, size=len(ready), replace=True)
    boot[b] = (np.median(off_b) / np.median(ready_b) - 1) * 100

ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"JVC 1-bed 2025 premium: {premium:.1f}%  (95% CI {ci_low:.1f}% to {ci_high:.1f}%)")
check_close("CI lower bound", ci_low,  15.6, tolerance=0.03)
check_close("CI upper bound", ci_high, 18.8, tolerance=0.03)

**9b (market-wide, per year):** now the question is about the *typical like-for-like premium across groups*, so we resample **groups**, not individual sales: draw the ~120 groups with replacement and recompute the weighted median each time. This is a **cluster bootstrap**, because the groups are the unit we're generalising over.

✏️ **TODO 9b:** draw `k` random group positions between 0 and `k − 1` with `rng.integers(0, k, size=k)`.

In [ ]:
rng = np.random.default_rng(SEED)
ci_rows = []
for year, g in comparable.groupby("year"):
    p, w, k = g["premium_pct"].to_numpy(), g["weight"].to_numpy(), len(g)
    boot = np.empty(N_BOOT)
    for b in range(N_BOOT):
        # TODO 9b: resample group positions
        idx = ___
        boot[b] = weighted_median(p[idx], w[idx])
    ci_rows.append({"year": year, "ci_low": np.percentile(boot, 2.5), "ci_high": np.percentile(boot, 97.5)})

lfl = lfl.drop(columns=["ci_low", "ci_high"], errors="ignore").join(pd.DataFrame(ci_rows).set_index("year"))
print(lfl[["groups", "like_for_like_premium_pct", "ci_low", "ci_high", "naive_premium_pct"]])
check_close("2025 CI lower bound", lfl.loc[2025, "ci_low"],  19.5, tolerance=0.05)
check_close("2025 CI upper bound", lfl.loc[2025, "ci_high"], 45.3, tolerance=0.05)

In [ ]:
fig, ax = plt.subplots()
yerr = [lfl["like_for_like_premium_pct"] - lfl["ci_low"], lfl["ci_high"] - lfl["like_for_like_premium_pct"]]
ax.errorbar(lfl.index, lfl["like_for_like_premium_pct"], yerr=yerr, fmt="o-", color="#C44E52", capsize=5,
            label="Like-for-like premium (95% bootstrap CI)")
ax.plot(lfl.index, lfl["naive_premium_pct"], "o--", color="#B0B0B0", label="Naive premium")
ax.set(title="Off-plan premium per sq ft over ready apartments in the same community and bedroom band",
       xlabel="", ylabel="Premium (%)", ylim=(0, None))
ax.title.set_fontsize(10)
ax.set_xticks(lfl.index, year_labels(lfl.index))
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(IMG_DIR / "q2_premium_by_year.png", dpi=150)
plt.show()

🤔 **Think about it:** the 2025 interval is noticeably *wider* than 2024's or 2026's, even though 2025 has the most groups. Why might a weighted median be unstable when a few groups carry a large share of the weight?

## Step 10: ⭐ Cross-check with a fixed-effects regression

🧠 **Why we're doing this**
If two different methods give the same answer, you can trust it more. Here's a standard econometric approach to the same 2025 question:

```
log(price_per_sqft) = β × off_plan + (one intercept per community-bedroom group) + error
```

- **`log` price:** a coefficient on log price is (approximately) a **percentage** effect, and `exp(β) − 1` is the exact premium.
- **`C(cell)`, group fixed effects:** one dummy per community × bedroom group, so β only compares off-plan with ready **inside the same group**. That's the regression version of Step 5's like-for-like idea.
- **Cluster-robust standard errors** (`cov_type="cluster"`): sales in the same group aren't independent of each other, so this makes the confidence interval honest instead of falsely narrow.

You used OLS with statsmodels in your DXB project. This is the same tool with fixed effects added.

✏️ **TODO 10:** the formula string: `"log_ppsf ~ off_plan + C(cell)"`.

In [ ]:
reg = d25.merge(tests[["community", "bedroom_band"]], on=["community", "bedroom_band"])   # the same 2025 groups
reg = reg.assign(log_ppsf=np.log(reg["price_per_sqft"]),
                 off_plan=reg["is_off_plan"].astype(int),
                 cell=reg["community"] + " | " + reg["bedroom_band"])

# TODO 10: the regression formula
model = smf.ols(___, data=reg).fit(cov_type="cluster", cov_kwds={"groups": reg["cell"]})

beta = model.params["off_plan"]
lo, hi = model.conf_int().loc["off_plan"]
print(f"Regression premium 2025: {np.exp(beta) - 1:.1%}  (95% CI {np.exp(lo) - 1:.1%} to {np.exp(hi) - 1:.1%}), n = {int(model.nobs):,}")
check_close("Regression premium (%)", (np.exp(beta) - 1) * 100, 32.9)

The regression lands at about **33%** with a tighter interval (28–38%), close to the weighted-median ~30%. Two methods with different assumptions agree on **roughly 30%**. That agreement is what lets you state the number with confidence.

**What the premium does *not* tell you.** Off-plan units are new; ready stock includes older buildings. Off-plan buyers also get payment plans. So the premium mixes *newness*, *financing terms* and *expectations*, and this data can't separate them. Say so.

## Step 11: Save the results

These go in `data/gold/`, small enough for Git, and the README and Power BI will use them. Just run it.

In [ ]:
comparable.to_csv(OUT_DIR / "q2_premium_by_group.csv", index=False)
lfl.reset_index().to_csv(OUT_DIR / "q2_premium_by_year.csv", index=False)
tests.to_csv(OUT_DIR / "q2_tests_2025.csv", index=False)
share.rename("off_plan_share_pct").reset_index().to_csv(OUT_DIR / "q2_off_plan_share.csv", index=False)

for f in ["q2_premium_by_group.csv", "q2_premium_by_year.csv", "q2_tests_2025.csv", "q2_off_plan_share.csv"]:
    print("✅", OUT_DIR / f)
for f in ["q2_naive_vs_like_for_like.png", "q2_premium_by_year.png"]:
    print("🖼 ", IMG_DIR / f)

## ✅ Module 5 complete

**What you built:** a full statistical answer to Q2. Share trend, naive vs like-for-like premium, a hypothesis test with an effect size, a corrected test across 120 groups, bootstrap confidence intervals and a regression cross-check, plus two README-ready charts.

**✏️ Write your findings:** fill in `docs/findings/q2_offplan_vs_ready.md` from your results and send it to me for review. Do the Q1 findings at the same time if you haven't yet.

**🎤 Practise saying these out loud** (model answers in `LEARNING.md`):
1. What does a p-value mean, and what doesn't it tell you?
2. Why Mann–Whitney instead of a t-test?
3. What's an effect size, and which one did you use?
4. What's the multiple comparisons problem, and how did you handle it?
5. Explain the bootstrap to a non-technical manager.
6. What is confounding? How did you deal with it here?
7. Your naive premium fell but your like-for-like premium rose. Explain.

**Commit your work**
```bash
git add python/05_q2_offplan_vs_ready.ipynb databricks/05_export_q2_extract.ipynb data/gold/ images/ docs/findings/q2_offplan_vs_ready.md
git commit -m "Add Q2 off-plan vs ready: like-for-like premium, Mann-Whitney + BH, bootstrap CIs, FE regression"
git push
```
Make sure `data/extracts/q2_apartments.parquet` does **not** appear in `git status`. It should be ignored.

**Next:** Module 6, Q5 sales forecast. Seasonality, a structural break (the 2026 slowdown), forecasting with backtesting, and an Excel `FORECAST.ETS` cross-check.